# Frame Picker
### Picks frames between two timestamps of both projection videos of a Single particle at a single power.

In [ ]:
import cv2
import os
from datetime import datetime

start = datetime.now()

particle = "C61"
power = 4
particle_file = f"{particle}y_40s_{power}mW"
projections = ["xz", "yz"]

video_loc = r"A:\LML\Videos"
output_loc = r"A:\LML\Images"

t1 = 0.0
t2 = 0.0

for proj in projections:
    print(f"\nProcessing {proj}...")
    video_path = f"{video_loc}\\{proj}\\{particle_file}.wmv"
    os.makedirs(output_loc, exist_ok=True)
    cap = cv2.VideoCapture(video_path)

    print("Opened:", cap.isOpened())
    print("FPS:", cap.get(cv2.CAP_PROP_FPS))
    print("Resolution:", int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), "x", int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))

    saved = 0
    printed_shape = False

    if t1 == 0 and t2 == 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0) # faster for t = 0 case

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if proj.lower() == "xz": # cropping out the right side for xz projection
            h, w = frame.shape[:2]
            if w < h:
                raise ValueError(f"{w}<{h}")
            frame = frame[:, :h]

            if not printed_shape:
                print("Cropped frame shape:", frame.shape)
                printed_shape = True

        timestamp = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0

        if t1 <= timestamp <= t2:
            filename = f"{particle}_{proj}_{power}mW_{timestamp:.3f}s.png"
            cv2.imwrite(os.path.join(output_loc, filename), frame)
            saved += 1

        if t1 == t2:
            break # break early when only one frame is needed

    cap.release()
    print(f"{saved} frames saved")

print(f"\nRuntime: {datetime.now()-start}")